In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LM(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.30501630902290344
epoch:  1, loss: 0.23218581080436707
epoch:  2, loss: 0.1589062511920929
epoch:  3, loss: 0.09207072854042053
epoch:  4, loss: 0.04405326768755913
epoch:  5, loss: 0.030184155330061913
epoch:  6, loss: 0.021635886281728745
epoch:  7, loss: 0.020236091688275337
epoch:  8, loss: 0.019496439024806023
epoch:  9, loss: 0.018621260300278664
epoch:  10, loss: 0.018608879297971725
epoch:  11, loss: 0.01729423739016056
epoch:  12, loss: 0.01646243967115879
epoch:  13, loss: 0.01593129336833954
epoch:  14, loss: 0.0155657809227705
epoch:  15, loss: 0.01465947087854147
epoch:  16, loss: 0.014104432426393032
epoch:  17, loss: 0.013753993436694145
epoch:  18, loss: 0.013375572860240936
epoch:  19, loss: 0.012608572840690613
epoch:  20, loss: 0.01221139170229435
epoch:  21, loss: 0.011985601857304573
epoch:  22, loss: 0.011578790843486786
epoch:  23, loss: 0.011104091070592403
epoch:  24, loss: 0.010874809697270393
epoch:  25, loss: 0.01074568834155798
epoch:  26

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9343100786209106
Test metrics:  R2 = 0.9254029989242554


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.LM(model=model,batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.38510462641716003
epoch:  1, loss: 0.0327775701880455
epoch:  2, loss: 0.03192806616425514
epoch:  3, loss: 0.029701506718993187
epoch:  4, loss: 0.028234733268618584
epoch:  5, loss: 0.027244385331869125
epoch:  6, loss: 0.020980076864361763
epoch:  7, loss: 0.01824868470430374
epoch:  8, loss: 0.01626700907945633
epoch:  9, loss: 0.014935638755559921
epoch:  10, loss: 0.014093643054366112
epoch:  11, loss: 0.013571233488619328
epoch:  12, loss: 0.01323216874152422
epoch:  13, loss: 0.012716417200863361
epoch:  14, loss: 0.011959473602473736
epoch:  15, loss: 0.011577296070754528
epoch:  16, loss: 0.011364052072167397
epoch:  17, loss: 0.011020016856491566
epoch:  18, loss: 0.010553023777902126
epoch:  19, loss: 0.010341471061110497
epoch:  20, loss: 0.010229822248220444
epoch:  21, loss: 0.009966403245925903
epoch:  22, loss: 0.00971048604696989
epoch:  23, loss: 0.009605432860553265
epoch:  24, loss: 0.009580912068486214
epoch:  25, loss: 0.00929106306284666
epoch

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9614063501358032
Test metrics:  R2 = 0.9348249435424805
